# 05 — Aggregation & Grouping

**What's in this notebook:**
- Aggregate functions — `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` — collapsing many rows into one
- `GROUP BY` — partitioning rows into groups and the single-value-per-group rule
- `HAVING` — filtering groups (and how it differs from `WHERE`)
- `NULL` in aggregates, and the three faces of `COUNT` — `COUNT(*)`, `COUNT(col)`, `COUNT(DISTINCT col)`
- Conditional aggregates with `FILTER (WHERE ...)`
- Grouping extensions — `ROLLUP`, `CUBE`, `GROUPING SETS` — multiple grouping levels in one query
- Common pitfalls to carry forward

This notebook continues using `customers`, `orders`, and `prospects` from earlier notebooks.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

## 1. Aggregate functions — collapsing many rows into one

An **aggregate function** takes a set of rows and returns a single value. The five standard aggregates cover almost everything:

- **`COUNT`** — how many
- **`SUM`** — total
- **`AVG`** — arithmetic mean
- **`MIN`** — smallest value
- **`MAX`** — largest value

When you call an aggregate without `GROUP BY`, the whole table is treated as one group, and you get a single-row result.

In [ ]:
%%sql
-- Whole-table aggregates: the entire orders table becomes one group.
SELECT COUNT(*)    AS order_count,
       SUM(amount) AS total_amount,
       AVG(amount) AS avg_amount,
       MIN(amount) AS min_amount,
       MAX(amount) AS max_amount
FROM orders;

## 2. GROUP BY — partitioning rows and the single-value-per-group rule

`GROUP BY <columns>` partitions the rows from `FROM`/`WHERE` into groups that share the same values for those columns, then produces **exactly one output row per group**.

That "one output row" constraint drives the single most important rule in aggregation:

> **Every column in `SELECT` must either be in `GROUP BY` or be inside an aggregate function.**

The reason is mechanical. Imagine `GROUP BY customer_id` collapses three of Alice's orders into one row. The output row clearly has `customer_id = 1` (that's the group). It clearly has `SUM(amount) = 179.49` (that's an aggregate). But what would it have for raw `amount` — the first order's? The largest? The aggregate engine refuses to guess.

Standard SQL enforces this rule. MySQL historically allowed it and silently picked an arbitrary value per group — fixed in MySQL 5.7+ with the `ONLY_FULL_GROUP_BY` mode.

In [ ]:
%%sql
-- Per-customer order summary. We LEFT JOIN so Carol (no orders) still appears.
SELECT c.customer_id,
       c.name,
       COUNT(o.order_id)          AS order_count,
       COALESCE(SUM(o.amount), 0) AS total_spent,
       AVG(o.amount)              AS avg_order_value
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC;

## 3. HAVING — filtering groups, not rows

`WHERE` filters **rows** before grouping. `HAVING` filters **groups** after grouping. The two are not interchangeable, and the choice matters for both correctness and performance.

Recall the execution order from notebook 02 §5:

```
  FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Because `WHERE` runs before `GROUP BY`, aggregates do not exist yet at `WHERE` time — `WHERE SUM(amount) > 100` is a category error. Because `HAVING` runs after `GROUP BY`, aggregates *do* exist there.

**Rule of thumb:** put a predicate in `WHERE` if you can. Filtering rows early shrinks the input to `GROUP BY` and is almost always faster than computing aggregates over the full table and then discarding most of them with `HAVING`.

In [ ]:
%%sql
-- WHERE filters individual orders; HAVING filters whole customer groups.
-- "Customers whose orders over $20 total more than $100."
SELECT c.name,
       COUNT(*)    AS qualifying_orders,
       SUM(amount) AS qualifying_total
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
WHERE amount > 20                          -- row filter, runs first
GROUP BY c.name
HAVING SUM(amount) > 100                   -- group filter, runs after GROUP BY
ORDER BY qualifying_total DESC;

## 4. NULL in aggregates, and the three faces of COUNT

A single rule covers most of it: **every aggregate except `COUNT(*)` ignores `NULL`.**

- `SUM(amount)` skips `NULL`s — they contribute nothing to the total.
- `AVG(amount)` divides the sum by the number of **non-`NULL`** values. So if one row out of ten has `NULL`, the average divides by 9. Usually what you want, occasionally surprising.
- `MIN` and `MAX` skip `NULL`s.

`COUNT` has three distinct forms, and confusing them is one of the most common SQL bugs:

- **`COUNT(*)`** — number of rows in the group. `NULL`s are counted because the row exists.
- **`COUNT(col)`** — number of rows where `col` is not `NULL`. Different from `COUNT(*)` whenever the column is nullable.
- **`COUNT(DISTINCT col)`** — number of distinct non-`NULL` values of `col`.

When every aggregate input is `NULL` (or there are no rows), `SUM` returns `NULL`, not 0. Wrap with `COALESCE(SUM(...), 0)` if you want zero.

In [ ]:
%%sql
-- The three COUNTs disagree on customers because email is nullable.
-- Carol has NULL email, so COUNT(email) = 2 even though COUNT(*) = 3.
SELECT COUNT(*)                AS total_rows,
       COUNT(email)            AS with_email,
       COUNT(DISTINCT email)   AS distinct_emails
FROM customers;

In [ ]:
%%sql
-- COUNT(DISTINCT customer_id) on orders: how many *unique* customers placed orders.
-- Alice has 2 orders, Bob has 1 — so 2 distinct customers, not 3 orders.
SELECT COUNT(*)                    AS orders_placed,
       COUNT(DISTINCT customer_id) AS customers_who_ordered
FROM orders;

## 5. Conditional aggregates with FILTER

Sometimes you want to aggregate only the rows that match a condition, *and* aggregate over a different condition in the same row, all in one pass. The standard SQL feature for this is `FILTER (WHERE ...)`, attached to an aggregate call:

```
SUM(amount) FILTER (WHERE order_date >= '2024-04-10')
```

This is dramatically cleaner than the old `SUM(CASE WHEN ... THEN amount END)` idiom, and it lets the engine optimize more aggressively. Supported by Postgres, DuckDB, SQLite (3.30+), and Snowflake. SQL Server and MySQL still require the `CASE` form.

It's the right tool when you want side-by-side aggregates over different slices of the same rows — for instance, this-period vs prior-period totals in a single row.

In [ ]:
%%sql
-- Two aggregates over different slices, in one row per customer.
SELECT c.name,
       SUM(o.amount) FILTER (WHERE o.amount < 50)  AS small_orders_total,
       SUM(o.amount) FILTER (WHERE o.amount >= 50) AS large_orders_total,
       SUM(o.amount)                               AS overall_total
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.name
ORDER BY overall_total DESC NULLS LAST;

## 6. Grouping extensions — ROLLUP, CUBE, GROUPING SETS

Sometimes you want **multiple grouping levels in one result** — a total per customer per year, plus a total per customer across all years, plus a grand total. You could write three separate queries and `UNION ALL` them, but standard SQL has three extensions that do it in one pass:

- **`GROUPING SETS ((a, b), (a), ())`** — the explicit form. Each tuple is one grouping level. The empty tuple `()` means "the whole table, no grouping" — the grand total.
- **`ROLLUP(a, b, c)`** — shorthand for the *hierarchical* grouping sets `(a, b, c), (a, b), (a), ()`. Adds subtotals along a hierarchy. Good for time hierarchies (year → quarter → month) or region hierarchies.
- **`CUBE(a, b, c)`** — the *power set*: every combination of the listed columns. Gives you all possible aggregation slices in one query.

Rows that come from a "rolled-up" level have `NULL` in the columns that were not part of that level. The `GROUPING(col)` function returns 1 for those synthetic `NULL`s and 0 for real values, so you can tell the two apart.

In [ ]:
%%sql
-- ROLLUP across (customer, year): per-customer-per-year, per-customer, and grand total.
-- GROUPING() distinguishes 'real NULL in data' from 'subtotal placeholder NULL'.
SELECT c.name,
       EXTRACT(YEAR FROM o.order_date)        AS order_year,
       SUM(o.amount)                          AS total,
       GROUPING(c.name)                       AS is_name_subtotal,
       GROUPING(EXTRACT(YEAR FROM o.order_date)) AS is_year_subtotal
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY ROLLUP (c.name, EXTRACT(YEAR FROM o.order_date))
ORDER BY c.name NULLS LAST, order_year NULLS LAST;

## 7. Common pitfalls (carry forward)

- **Forgetting the single-value rule** — every `SELECT` column outside an aggregate must be in `GROUP BY`. Strict engines reject the query; MySQL historically picked an arbitrary value, which silently produces wrong answers.
- **Using `WHERE` to filter aggregates** — aggregates don't exist yet at `WHERE` time. Use `HAVING` for aggregate predicates, and keep row-level predicates in `WHERE` where they belong.
- **`COUNT(*)` vs `COUNT(col)` confusion** — they're equal only when `col` is `NOT NULL`. If you're counting "how many records have a non-empty value of `col`", use `COUNT(col)`.
- **`SUM` returns `NULL` over zero non-`NULL` rows, not 0.** Wrap with `COALESCE(SUM(...), 0)` if downstream code expects a number.
- **`AVG` divides by the non-`NULL` count, not the row count.** If `NULL`s should count as zero, write `AVG(COALESCE(col, 0))` explicitly.
- **`COUNT(DISTINCT col)` on a huge table is expensive.** It can't use the same one-pass counting strategy as plain `COUNT` and forces a sort or hash. For approximate counts, most modern engines offer `APPROX_COUNT_DISTINCT` (DuckDB, BigQuery, Snowflake) or `count_distinct_approx` (Spark).
- **Aggregating across a `LEFT JOIN` then filtering the right side in `WHERE`** — same trap as notebook 04. Use a predicate in the `ON` clause if you want to keep unmatched left rows.

Next up: **notebook 06 — Subqueries & CTEs**, where queries start nesting inside other queries and `WITH` becomes your best friend.